# ⭐ Day 77: Production-Ready AI & MLOps
From Notebook to Deployed Model

**Day 77 of 369-day Python & AI Learning Path** 🚀


## Introduction

Building a powerful machine learning model in a Jupyter notebook is an incredible achievement, but it represents only the first half of the journey. In the real world, the true value of AI emerges when models are reliably deployed, served, and maintained in production environments. Today, we bridge the gap between experimental data science and robust software engineering.

MLOps—Machine Learning Operations—is the discipline of streamlining the process of taking machine learning models to production and then maintaining and monitoring them. It borrows principles from DevOps but addresses the unique challenges of ML: data drift, model degradation, reproducibility, and the need for continuous retraining. Without MLOps, even the most accurate model remains trapped in a notebook, unable to deliver business value.

In this comprehensive lesson, we will explore the complete lifecycle of a production ML system. We will learn how to serialize and version models using industry-standard tools, build robust inference pipelines that handle real-world data gracefully, and create reusable prediction services. We will examine how to expose models via REST APIs using Flask and build interactive demos with Streamlit. Finally, we will touch upon monitoring, logging, and containerization with Docker—essential skills for any AI engineer aiming to deploy scalable, maintainable systems.

By the end of Day 77, you will possess the knowledge and practical code templates to transform your notebook experiments into production-grade AI services. Let's turn your models from static artifacts into dynamic, deployed intelligence. 🚀


## 📋 Table of Contents

1. [What is MLOps and Why It Is Essential](#1-what-is-mlops-and-why-it-is-essential)
2. [Model Serialization and Versioning](#2-model-serialization-and-versioning)
3. [Building a Robust Inference Pipeline](#3-building-a-robust-inference-pipeline)
4. [Creating a Prediction Service Class](#4-creating-a-prediction-service-class)
5. [Simple REST API with Flask](#5-simple-rest-api-with-flask)
6. [Streamlit Demo for Interactive Model Serving](#6-streamlit-demo-for-interactive-model-serving)
7. [Model Monitoring and Logging Basics](#7-model-monitoring-and-logging-basics)
8. [Containerization Concepts](#8-containerization-concepts)
9. [End-to-End Example: Train → Save → Serve](#9-end-to-end-example-train--save--serve-pipeline)
10. [Best Practices for Production ML Systems](#10-best-practices-for-production-ml-systems)
11. [Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Production Tips](#-solutions--production-tips)


## 1. What is MLOps and Why It Is Essential

MLOps (Machine Learning Operations) is a set of practices that aims to deploy and maintain ML models in production reliably and efficiently. It combines ML, DevOps, and Data Engineering.

**Key Challenges MLOps Solves:**
- **Reproducibility**: Ensuring experiments can be replicated exactly
- **Versioning**: Tracking models, data, and code versions
- **Scalability**: Serving models to thousands or millions of users
- **Monitoring**: Detecting model drift and performance degradation
- **Governance**: Managing compliance and audit trails

**The MLOps Lifecycle:**
1. Data Collection & Preparation
2. Model Training & Experimentation
3. Model Evaluation & Validation
4. Model Deployment & Serving
5. Monitoring & Maintenance
6. Retraining & Iteration


## 2. Model Serialization and Versioning

Before deployment, we must save (serialize) our trained models. Python offers several robust options for this task.


In [1]:
# Cell 1: Setup and Imports
import numpy as np
import pandas as pd
import joblib
import pickle
import json
import os
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Ensure output directory exists
os.makedirs('models', exist_ok=True)
print("✅ Environment ready for MLOps!")


✅ Environment ready for MLOps!


In [2]:
# Cell 2: Create Sample Dataset and Train a Model
# Using the familiar mental health / screen time dataset structure
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'age': np.random.randint(18, 65, n_samples),
    'daily_screen_time_min': np.random.normal(300, 120, n_samples).astype(int),
    'social_media_time_min': np.random.normal(120, 60, n_samples).astype(int),
    'sleep_hours': np.random.normal(7, 1.5, n_samples),
    'physical_activity_min': np.random.normal(45, 30, n_samples).astype(int),
    'negative_interactions_count': np.random.poisson(3, n_samples),
    'positive_interactions_count': np.random.poisson(8, n_samples),
    'anxiety_level': np.random.randint(1, 11, n_samples),
    'mood_level': np.random.randint(1, 11, n_samples)
})

# Create synthetic target: stress_level (0=Low, 1=Medium, 2=High)
data['stress_level'] = (
    (data['daily_screen_time_min'] > 400).astype(int) +
    (data['sleep_hours'] < 6).astype(int) +
    (data['anxiety_level'] > 7).astype(int)
).clip(0, 2)

X = data.drop('stress_level', axis=1)
y = data['stress_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"✅ Model trained! Test Accuracy: {accuracy:.4f}")


✅ Model trained! Test Accuracy: 0.9850


In [3]:
# Cell 3: Save Model with joblib (Recommended for sklearn)
model_filename = f"models/stress_predictor_v1_{datetime.now().strftime('%Y%m%d')}.joblib"
joblib.dump(model, model_filename)
print(f"📦 Model saved to: {model_filename}")
print(f"📁 File size: {os.path.getsize(model_filename) / 1024:.2f} KB")


📦 Model saved to: models/stress_predictor_v1_20260428.joblib
📁 File size: 716.30 KB


In [4]:
# Cell 4: Save Complete Pipeline (Model + Preprocessor)
# In production, we often need to save the scaler/preprocessor too
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pipeline = {
    'model': model,
    'scaler': scaler,
    'feature_names': list(X.columns),
    'training_date': datetime.now().isoformat(),
    'model_version': '1.0.0',
    'metrics': {'accuracy': accuracy}
}

pipeline_filename = 'models/stress_pipeline_v1.joblib'
joblib.dump(pipeline, pipeline_filename)
print(f"📦 Complete pipeline saved to: {pipeline_filename}")


📦 Complete pipeline saved to: models/stress_pipeline_v1.joblib


In [5]:
# Cell 5: Model Versioning with a Simple Registry
# A lightweight versioning system before adopting MLflow
class ModelRegistry:
    def __init__(self, registry_path='models/registry.json'):
        self.registry_path = registry_path
        self.registry = self._load_registry()
    
    def _load_registry(self):
        if os.path.exists(self.registry_path):
            with open(self.registry_path, 'r') as f:
                return json.load(f)
        return {'models': []}
    
    def register_model(self, name, version, filepath, metrics, tags=None):
        entry = {
            'name': name,
            'version': version,
            'filepath': filepath,
            'registered_at': datetime.now().isoformat(),
            'metrics': metrics,
            'tags': tags or {},
            'status': 'active'
        }
        self.registry['models'].append(entry)
        self._save_registry()
        print(f"✅ Registered {name} v{version}")
        return entry
    
    def _save_registry(self):
        with open(self.registry_path, 'w') as f:
            json.dump(self.registry, f, indent=2)
    
    def get_latest(self, name):
        models = [m for m in self.registry['models'] if m['name'] == name and m['status'] == 'active']
        return sorted(models, key=lambda x: x['registered_at'], reverse=True)[0] if models else None

# Register our model
registry = ModelRegistry()
registry.register_model(
    name='stress_predictor',
    version='1.0.0',
    filepath=pipeline_filename,
    metrics={'accuracy': float(accuracy)},
    tags={'framework': 'sklearn', 'type': 'random_forest'}
)


✅ Registered stress_predictor v1.0.0


{'name': 'stress_predictor',
 'version': '1.0.0',
 'filepath': 'models/stress_pipeline_v1.joblib',
 'registered_at': '2026-04-28T06:12:15.752843',
 'metrics': {'accuracy': 0.985},
 'tags': {'framework': 'sklearn', 'type': 'random_forest'},
 'status': 'active'}

## 3. Building a Robust Inference Pipeline

Production inference must handle edge cases, validate inputs, and return structured outputs. Never assume inputs are clean!


In [6]:
# Cell 6: Input Validation and Preprocessing for Inference
class InputValidator:
    REQUIRED_FEATURES = [
        'age', 'daily_screen_time_min', 'social_media_time_min',
        'sleep_hours', 'physical_activity_min', 'negative_interactions_count',
        'positive_interactions_count', 'anxiety_level', 'mood_level'
    ]
    
    @classmethod
    def validate(cls, input_data):
        errors = []
        
        # Check required fields
        missing = set(cls.REQUIRED_FEATURES) - set(input_data.keys())
        if missing:
            errors.append(f"Missing required fields: {missing}")
        
        # Check data types and ranges
        for field in cls.REQUIRED_FEATURES:
            if field in input_data:
                val = input_data[field]
                if not isinstance(val, (int, float)):
                    errors.append(f"{field} must be numeric, got {type(val).__name__}")
                elif val < 0:
                    errors.append(f"{field} cannot be negative")
        
        if errors:
            raise ValueError("; ".join(errors))
        
        # Return ordered features as array
        return np.array([[input_data[f] for f in cls.REQUIRED_FEATURES]])

# Test validation
test_input = {
    'age': 25,
    'daily_screen_time_min': 450,
    'social_media_time_min': 180,
    'sleep_hours': 5.5,
    'physical_activity_min': 20,
    'negative_interactions_count': 5,
    'positive_interactions_count': 3,
    'anxiety_level': 8,
    'mood_level': 3
}

validated = InputValidator.validate(test_input)
print(f"✅ Input validated. Shape: {validated.shape}")


✅ Input validated. Shape: (1, 9)


## 4. Creating a Prediction Service Class

A well-designed `Predictor` class encapsulates all inference logic, making it reusable across APIs, batch jobs, and streaming services.


In [7]:
# Cell 7: Production-Ready Predictor Class
class StressPredictor:
    """
    Production-ready prediction service for stress level classification.
    Handles model loading, input validation, preprocessing, and prediction.
    """
    
    def __init__(self, model_path):
        self.model_path = model_path
        self.pipeline = None
        self.model = None
        self.scaler = None
        self.feature_names = None
        self.version = None
        self._load_model()
    
    def _load_model(self):
        """Load the complete pipeline from disk."""
        if not os.path.exists(self.model_path):
            raise FileNotFoundError(f"Model file not found: {self.model_path}")
        
        self.pipeline = joblib.load(self.model_path)
        self.model = self.pipeline['model']
        self.scaler = self.pipeline['scaler']
        self.feature_names = self.pipeline['feature_names']
        self.version = self.pipeline.get('model_version', 'unknown')
        print(f"✅ Model loaded (version: {self.version})")
    
    def predict(self, input_data):
        """
        Make a prediction with full validation and error handling.
        
        Args:
            input_data: dict or list of dicts with feature values
            
        Returns:
            dict with predictions, probabilities, and metadata
        """
        try:
            # Handle single input
            is_single = isinstance(input_data, dict)
            if is_single:
                input_data = [input_data]
            
            # Validate all inputs
            features = []
            for item in input_data:
                validated = InputValidator.validate(item)
                features.append(validated[0])
            
            features = np.array(features)
            
            # Preprocess
            features_scaled = self.scaler.transform(features)
            
            # Predict
            predictions = self.model.predict(features_scaled)
            probabilities = self.model.predict_proba(features_scaled)
            
            # Build response
            stress_labels = {0: 'Low', 1: 'Medium', 2: 'High'}
            results = []
            for i, pred in enumerate(predictions):
                results.append({
                    'stress_level': int(pred),
                    'stress_label': stress_labels[int(pred)],
                    'confidence': float(np.max(probabilities[i])),
                    'probabilities': {
                        stress_labels[j]: float(prob) 
                        for j, prob in enumerate(probabilities[i])
                    }
                })
            
            return {
                'success': True,
                'model_version': self.version,
                'predictions': results[0] if is_single else results,
                'timestamp': datetime.now().isoformat()
            }
            
        except Exception as e:
            return {
                'success': False,
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            }
    
    def health_check(self):
        """Return model health status."""
        return {
            'status': 'healthy',
            'model_version': self.version,
            'model_path': self.model_path,
            'features_expected': self.feature_names,
            'timestamp': datetime.now().isoformat()
        }

# Instantiate predictor
predictor = StressPredictor(pipeline_filename)
print(predictor.health_check())


✅ Model loaded (version: 1.0.0)
{'status': 'healthy', 'model_version': '1.0.0', 'model_path': 'models/stress_pipeline_v1.joblib', 'features_expected': ['age', 'daily_screen_time_min', 'social_media_time_min', 'sleep_hours', 'physical_activity_min', 'negative_interactions_count', 'positive_interactions_count', 'anxiety_level', 'mood_level'], 'timestamp': '2026-04-28T06:12:15.896063'}


In [8]:
# Cell 8: Test the Predictor
result = predictor.predict(test_input)
print(json.dumps(result, indent=2))


{
  "success": true,
  "model_version": "1.0.0",
  "predictions": {
    "stress_level": 1,
    "stress_label": "Medium",
    "confidence": 0.73,
    "probabilities": {
      "Low": 0.06,
      "Medium": 0.73,
      "High": 0.21
    }
  },
  "timestamp": "2026-04-28T06:12:15.944501"
}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## 5. Simple REST API with Flask

Flask provides a lightweight way to expose your model as a REST API. Below is the complete structure for a production-style prediction endpoint.


In [9]:
# Cell 9: Flask API Structure (save as app.py in production)
flask_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np
from datetime import datetime
import os

app = Flask(__name__)

# Load model at startup (singleton pattern)
MODEL_PATH = os.getenv('MODEL_PATH', 'models/stress_pipeline_v1.joblib')
predictor = None

def load_model():
    global predictor
    pipeline = joblib.load(MODEL_PATH)
    predictor = {
        'model': pipeline['model'],
        'scaler': pipeline['scaler'],
        'feature_names': pipeline['feature_names'],
        'version': pipeline.get('model_version', '1.0.0')
    }
    print(f"✅ Model v{predictor['version']} loaded successfully")

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'model_version': predictor['version'],
        'timestamp': datetime.now().isoformat()
    })

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json(force=True)
        
        if not data:
            return jsonify({'success': False, 'error': 'No input data provided'}), 400
        
        # Validate input
        required = predictor['feature_names']
        missing = set(required) - set(data.keys())
        if missing:
            return jsonify({'success': False, 'error': f'Missing fields: {missing}'}), 400
        
        # Extract and scale features
        features = np.array([[data[f] for f in required]])
        features_scaled = predictor['scaler'].transform(features)
        
        # Predict
        prediction = predictor['model'].predict(features_scaled)[0]
        probabilities = predictor['model'].predict_proba(features_scaled)[0]
        
        labels = {0: 'Low', 1: 'Medium', 2: 'High'}
        
        return jsonify({
            'success': True,
            'model_version': predictor['version'],
            'prediction': {
                'stress_level': int(prediction),
                'stress_label': labels[int(prediction)],
                'confidence': float(np.max(probabilities)),
                'probabilities': {labels[i]: float(p) for i, p in enumerate(probabilities)}
            },
            'timestamp': datetime.now().isoformat()
        })
        
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    try:
        data = request.get_json(force=True)
        records = data.get('records', [])
        
        if not records:
            return jsonify({'success': False, 'error': 'No records provided'}), 400
        
        results = []
        for record in records:
            features = np.array([[record.get(f, 0) for f in predictor['feature_names']]])
            features_scaled = predictor['scaler'].transform(features)
            pred = predictor['model'].predict(features_scaled)[0]
            results.append({'stress_level': int(pred), 'stress_label': ['Low', 'Medium', 'High'][int(pred)]})
        
        return jsonify({'success': True, 'predictions': results, 'count': len(results)})
        
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

if __name__ == '__main__':
    load_model()
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

with open('models/flask_app.py', 'w') as f:
    f.write(flask_code)
print("📦 Flask app structure saved to models/flask_app.py")
print("💡 To run: python models/flask_app.py")


📦 Flask app structure saved to models/flask_app.py
💡 To run: python models/flask_app.py


## 6. Streamlit Demo for Interactive Model Serving

Streamlit is perfect for rapid prototyping and internal demos. It allows stakeholders to interact with models without writing API client code.


In [10]:
# Cell 10: Streamlit App Structure (save as streamlit_app.py)
streamlit_code = '''
import streamlit as st
import joblib
import numpy as np
from datetime import datetime
import os

st.set_page_config(page_title="Stress Predictor", page_icon="🧠", layout="centered")

@st.cache_resource
def load_model():
    pipeline = joblib.load('models/stress_pipeline_v1.joblib')
    return pipeline

pipeline = load_model()
model = pipeline['model']
scaler = pipeline['scaler']
feature_names = pipeline['feature_names']
version = pipeline.get('model_version', '1.0.0')

st.title("🧠 Stress Level Predictor")
st.markdown(f"*Model version: {version}*")
st.markdown("---")

with st.form("prediction_form"):
    st.subheader("📋 Input Features")
    
    col1, col2 = st.columns(2)
    
    with col1:
        age = st.number_input("Age", min_value=18, max_value=100, value=30)
        screen_time = st.number_input("Daily Screen Time (min)", min_value=0, max_value=1440, value=300)
        social_time = st.number_input("Social Media Time (min)", min_value=0, max_value=1440, value=120)
        sleep = st.number_input("Sleep Hours", min_value=0.0, max_value=24.0, value=7.0, step=0.5)
        activity = st.number_input("Physical Activity (min)", min_value=0, max_value=1440, value=45)
    
    with col2:
        neg_interactions = st.number_input("Negative Interactions", min_value=0, max_value=50, value=3)
        pos_interactions = st.number_input("Positive Interactions", min_value=0, max_value=50, value=8)
        anxiety = st.slider("Anxiety Level (1-10)", 1, 10, 5)
        mood = st.slider("Mood Level (1-10)", 1, 10, 7)
    
    submitted = st.form_submit_button("🔮 Predict Stress Level")

if submitted:
    input_data = {
        'age': age, 'daily_screen_time_min': screen_time,
        'social_media_time_min': social_time, 'sleep_hours': sleep,
        'physical_activity_min': activity, 'negative_interactions_count': neg_interactions,
        'positive_interactions_count': pos_interactions, 'anxiety_level': anxiety,
        'mood_level': mood
    }
    
    features = np.array([[input_data[f] for f in feature_names]])
    features_scaled = scaler.transform(features)
    
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    
    labels = {0: 'Low 🟢', 1: 'Medium 🟡', 2: 'High 🔴'}
    colors = {0: 'green', 1: 'orange', 2: 'red'}
    
    st.markdown("---")
    st.subheader("📊 Prediction Result")
    
    result_col, prob_col = st.columns(2)
    
    with result_col:
        st.markdown(f"### Stress Level: **{labels[int(prediction)]}**")
        st.markdown(f"**Confidence:** {np.max(probabilities):.2%}")
    
    with prob_col:
        st.markdown("**Probability Distribution:**")
        for i, label in enumerate(['Low', 'Medium', 'High']):
            st.progress(float(probabilities[i]), text=f"{label}: {probabilities[i]:.2%}")
    
    st.markdown(f"*Prediction made at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*")
'''

with open('models/streamlit_app.py', 'w') as f:
    f.write(streamlit_code)
print("📦 Streamlit app saved to models/streamlit_app.py")
print("💡 To run: streamlit run models/streamlit_app.py")


📦 Streamlit app saved to models/streamlit_app.py
💡 To run: streamlit run models/streamlit_app.py


## 7. Model Monitoring and Logging Basics

Production systems must log predictions, errors, and performance metrics. This enables debugging, audit trails, and drift detection.


In [11]:
# Cell 11: Production Logging Setup
import logging
from logging.handlers import RotatingFileHandler
import sys

def setup_logger(name='mlops', log_file='models/prediction.log'):
    """Configure production-grade logging with rotation."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    if logger.handlers:
        return logger
    
    # Formatter
    formatter = logging.Formatter(
        '%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Console handler
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.INFO)
    console.setFormatter(formatter)
    
    # Rotating file handler (10MB per file, keep 5 backups)
    file_handler = RotatingFileHandler(log_file, maxBytes=10*1024*1024, backupCount=5)
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)
    
    logger.addHandler(console)
    logger.addHandler(file_handler)
    
    return logger

logger = setup_logger()
logger.info("🚀 Logging system initialized")


2026-04-28 06:12:16 | INFO     | mlops | 🚀 Logging system initialized


In [12]:
# Cell 12: Prediction Logger for Monitoring
class PredictionLogger:
    def __init__(self, logger_name='mlops'):
        self.logger = logging.getLogger(logger_name)
        self.prediction_count = 0
        self.error_count = 0
    
    def log_prediction(self, input_data, result, latency_ms):
        self.prediction_count += 1
        self.logger.info(
            f"PREDICTION | count={self.prediction_count} | "
            f"input_keys={list(input_data.keys())} | "
            f"prediction={result.get('predictions', {}).get('stress_label', 'N/A')} | "
            f"confidence={result.get('predictions', {}).get('confidence', 0):.4f} | "
            f"latency_ms={latency_ms:.2f}"
        )
    
    def log_error(self, error_msg, input_data=None):
        self.error_count += 1
        self.logger.error(f"ERROR | count={self.error_count} | error={error_msg} | input={input_data}")
    
    def get_stats(self):
        return {
            'total_predictions': self.prediction_count,
            'total_errors': self.error_count,
            'error_rate': self.error_count / max(self.prediction_count, 1),
            'timestamp': datetime.now().isoformat()
        }

# Test logging
pred_logger = PredictionLogger()
import time
start = time.time()
result = predictor.predict(test_input)
latency = (time.time() - start) * 1000
pred_logger.log_prediction(test_input, result, latency)
print(f"✅ Logged prediction. Stats: {pred_logger.get_stats()}")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


2026-04-28 06:12:16 | INFO     | mlops | PREDICTION | count=1 | input_keys=['age', 'daily_screen_time_min', 'social_media_time_min', 'sleep_hours', 'physical_activity_min', 'negative_interactions_count', 'positive_interactions_count', 'anxiety_level', 'mood_level'] | prediction=Medium | confidence=0.7300 | latency_ms=24.17
✅ Logged prediction. Stats: {'total_predictions': 1, 'total_errors': 0, 'error_rate': 0.0, 'timestamp': '2026-04-28T06:12:16.156223'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## 8. Containerization Concepts

Docker containers ensure your model runs identically across development, staging, and production environments.


In [13]:
# Cell 13: Dockerfile Structure
dockerfile_content = '''
# Use official Python slim image
FROM python:3.10-slim

# Set working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (for layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model artifacts and application code
COPY models/ ./models/
COPY app.py .

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:5000/health || exit 1

# Run the application
CMD ["python", "app.py"]
'''

requirements_content = '''
flask==2.3.3
scikit-learn==1.3.0
numpy==1.24.3
pandas==2.0.3
joblib==1.3.2
gunicorn==21.2.0
'''

with open('models/Dockerfile', 'w') as f:
    f.write(dockerfile_content.strip())
    
with open('models/requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("📦 Dockerfile and requirements.txt created")
print("💡 Build: docker build -t stress-predictor .")
print("💡 Run: docker run -p 5000:5000 stress-predictor")


📦 Dockerfile and requirements.txt created
💡 Build: docker build -t stress-predictor .
💡 Run: docker run -p 5000:5000 stress-predictor


## 9. End-to-End Example: Train → Save → Serve Pipeline

Let's put it all together in a single, cohesive workflow.


In [14]:
# Cell 14: Complete End-to-End Pipeline
class MLPipeline:
    """
    Complete ML pipeline: Train, Evaluate, Save, and Serve.
    """
    
    def __init__(self, model_dir='models'):
        self.model_dir = model_dir
        os.makedirs(model_dir, exist_ok=True)
        self.pipeline = {}
        self.registry = ModelRegistry(os.path.join(model_dir, 'registry.json'))
    
    def train(self, X, y, test_size=0.2):
        """Train and evaluate the model."""
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train_s, y_train)
        
        accuracy = accuracy_score(y_test, model.predict(X_test_s))
        
        self.pipeline = {
            'model': model,
            'scaler': scaler,
            'feature_names': list(X.columns),
            'training_date': datetime.now().isoformat(),
            'model_version': '1.0.0',
            'metrics': {'accuracy': accuracy, 'test_size': test_size}
        }
        
        print(f"✅ Training complete. Accuracy: {accuracy:.4f}")
        return self
    
    def save(self, name='stress_pipeline'):
        """Save pipeline and register it."""
        version = self.pipeline['model_version']
        filename = f"{self.model_dir}/{name}_v{version}.joblib"
        joblib.dump(self.pipeline, filename)
        
        self.registry.register_model(
            name=name,
            version=version,
            filepath=filename,
            metrics=self.pipeline['metrics'],
            tags={'trained_at': self.pipeline['training_date']}
        )
        
        print(f"📦 Pipeline saved and registered: {filename}")
        return self
    
    def serve(self, input_data):
        """Run inference using the trained pipeline."""
        if not self.pipeline:
            raise ValueError("Pipeline not trained yet!")
        
        features = np.array([[input_data[f] for f in self.pipeline['feature_names']]])
        features_s = self.pipeline['scaler'].transform(features)
        pred = self.pipeline['model'].predict(features_s)[0]
        
        return {'stress_level': int(pred), 'label': ['Low', 'Medium', 'High'][int(pred)]}

# Execute full pipeline
pipeline = MLPipeline()
pipeline.train(X, y).save()
result = pipeline.serve(test_input)
print(f"🚀 End-to-end result: {result}")


✅ Training complete. Accuracy: 0.9850
✅ Registered stress_pipeline v1.0.0
📦 Pipeline saved and registered: models/stress_pipeline_v1.0.0.joblib
🚀 End-to-end result: {'stress_level': 2, 'label': 'High'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 10. Best Practices for Production ML Systems

**✅ DO:**
- Version everything: models, data, code, and configurations
- Validate all inputs rigorously before inference
- Use structured logging for every prediction and error
- Separate training and inference pipelines
- Implement health checks and readiness probes
- Design for graceful degradation when models fail
- Monitor data drift and model performance over time
- Use containerization for environment consistency
- Keep models small and fast for low-latency serving
- Document APIs with OpenAPI/Swagger specifications

**❌ DON'T:**
- Load models inside prediction functions (load once at startup)
- Expose raw exception messages to API consumers
- Hardcode paths or configurations
- Ignore memory leaks in long-running services
- Deploy without automated tests for the inference pipeline
- Forget to handle batch predictions efficiently


## 🛠️ Hands-On Exercises

Complete these exercises to solidify your MLOps skills. Each exercise builds upon the previous one.


### Exercise 1: Save and Load a Trained Model
Train a simple model (Logistic Regression or Random Forest) on any dataset, save it using `joblib`, then load it back and verify predictions match.


### Exercise 2: Build a Production-Ready Prediction Class
Create a `Predictor` class that encapsulates model loading, input validation, preprocessing, and prediction. Include proper error handling and return structured JSON-like responses.


### Exercise 3: Create a Simple Flask Prediction Endpoint
Write a Flask application with a `/predict` endpoint that accepts JSON input, validates it, runs inference using your Predictor class, and returns JSON results.


### Exercise 4: Add Input Validation and Error Handling
Enhance your Flask API to return appropriate HTTP status codes (400 for bad input, 500 for server errors) and descriptive error messages. Never expose internal stack traces.


### Exercise 5: Implement Basic Logging
Add Python `logging` to your prediction service. Log every prediction request with input summary, output, and latency. Also log all errors with full context.


### Exercise 6: Prepare a Model for Containerized Deployment
Write a `Dockerfile` and `requirements.txt` for your Flask prediction service. Ensure the model is loaded at container startup, not per request.


### Exercise 7: Design a Simple Model Versioning Strategy
Implement a JSON-based model registry that tracks model versions, file paths, training metrics, and active status. Write functions to register new models and retrieve the latest active version.


### Exercise 8: Build a Batch Prediction Endpoint
Create a `/batch_predict` endpoint that accepts a list of records and returns predictions for all of them efficiently. Compare the runtime of batch vs. single predictions.


### Exercise 9: Create a Streamlit Dashboard
Build a Streamlit app that lets users input features via sliders and buttons, displays predictions with confidence scores, and shows a history of recent predictions.


### Exercise 10: Implement a Health Check and Metrics Endpoint
Add `/health` and `/metrics` endpoints to your Flask app. The metrics endpoint should return total predictions, error counts, and average latency over the service lifetime.


## Solutions & Production Tips (Review After Attempting)

Below are detailed solutions and best-practice tips for each exercise.


### Solution 1: Save and Load
Always use `joblib` for sklearn models (handles numpy arrays better than pickle). Include the training timestamp and feature names in the saved object for future reference.


In [15]:
# Solution 1 Example
from sklearn.linear_model import LogisticRegression

# Train
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# Save with metadata
artifact = {'model': lr, 'features': list(X.columns), 'date': datetime.now().isoformat()}
joblib.dump(artifact, 'models/lr_model.joblib')

# Load and verify
loaded = joblib.load('models/lr_model.joblib')
assert list(X.columns) == loaded['features']
print("✅ Model save/load verified")


✅ Model save/load verified


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Solution 2: Prediction Class
Key principles: load model in `__init__`, never in `predict()`. Use `@staticmethod` or `@classmethod` for validators. Return dictionaries, not raw arrays. Catch ALL exceptions and return structured error responses.


In [16]:
# Solution 2 Example Structure
class MyPredictor:
    def __init__(self, path):
        self.artifact = joblib.load(path)
        self.model = self.artifact['model']
        self.scaler = self.artifact.get('scaler')
    
    def predict(self, data):
        try:
            # validate, preprocess, predict, format
            return {'success': True, 'prediction': pred}
        except Exception as e:
            return {'success': False, 'error': str(e)}


### Solution 3 & 4: Flask API
Use `request.get_json(force=True)` but always validate structure. Return tuples `(jsonify(response), status_code)` from Flask. Use a global singleton for the model to avoid reloading per request.


### Solution 5: Logging
Use `logging.getLogger()` with unique names. Use `RotatingFileHandler` to prevent disk overflow. Log structured data (JSON) if you plan to use log aggregation tools like ELK or Datadog.


### Solution 6: Docker
Multi-stage builds can reduce image size. Always use `.dockerignore` to exclude training data and notebooks. Set `PYTHONUNBUFFERED=1` to see logs immediately.


In [17]:
# .dockerignore example
dockerignore = '''
__pycache__/
*.pyc
*.pyo
.git/
notebooks/
data/
*.ipynb
'''
with open('models/.dockerignore', 'w') as f:
    f.write(dockerignore.strip())
print("✅ .dockerignore created")


✅ .dockerignore created


### Solution 7: Versioning
Use semantic versioning (MAJOR.MINOR.PATCH). Store a `metadata.json` alongside each model. Consider using MLflow or DVC for enterprise-grade versioning.


### Solution 8: Batch Prediction
Vectorize operations using numpy arrays rather than Python loops. If using sklearn, pass the entire batch matrix to `predict()` at once—it is significantly faster than looping.


In [18]:
# Solution 8: Vectorized batch prediction
batch_data = [test_input for _ in range(100)]
features = np.array([[d[f] for f in predictor.feature_names] for d in batch_data])
features_s = predictor.scaler.transform(features)
preds = predictor.model.predict(features_s)
print(f"✅ Batch of 100 predictions completed. Unique values: {np.unique(preds)}")


✅ Batch of 100 predictions completed. Unique values: [1]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


### Solution 9: Streamlit
Use `st.session_state` to persist prediction history across interactions. Use `st.cache_resource` for model loading. Add `st.spinner()` during inference for better UX.


### Solution 10: Monitoring
Use `time.time()` to measure latency. Store counters in a simple dictionary or use Prometheus client libraries for production. The `/metrics` endpoint should return machine-parseable formats.
